In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install transformers peft accelerate bitsandbytes

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, PeftConfig


adapter_path = '/content/drive/MyDrive/ollama_model/gold/'

try:
    config = PeftConfig.from_pretrained(adapter_path)
    base_model_name_or_path = config.base_model_name_or_path
    print(f"Detected base model from adapter_config.json: {base_model_name_or_path}")
except Exception as e:
    base_model_name_or_path = "HuggingFaceH4/zephyr-7b-beta"

tokenizer = AutoTokenizer.from_pretrained(base_model_name_or_path)

if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

print(f"Loading base model: {base_model_name_or_path}...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name_or_path,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
print("Base model loaded.")

if tokenizer.pad_token_id is not None and len(tokenizer) > base_model.config.vocab_size:
    base_model.resize_token_embeddings(len(tokenizer))


print(f"Loading LoRA adapter from: {adapter_path}...")
model = PeftModel.from_pretrained(base_model, adapter_path)
print("LoRA adapter loaded and merged with base model.")

model = model.merge_and_unload()
print("Adapter merged and unloaded from PEFT structure.")

model.eval()

print("Model (base + LoRA adapter) successfully loaded and ready for inference!")

Detected base model from adapter_config.json: HuggingFaceH4/zephyr-7b-beta


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading base model: HuggingFaceH4/zephyr-7b-beta...


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Base model loaded.
Loading LoRA adapter from: /content/drive/MyDrive/ollama_model/gold/...
LoRA adapter loaded and merged with base model.


/usr/local/lib/python3.11/dist-packages/peft/tuners/lora/bnb.py:351: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Adapter merged and unloaded from PEFT structure.
Model (base + LoRA adapter) successfully loaded and ready for inference!


In [30]:
import torch
from transformers import GenerationConfig, AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import re

if tokenizer.pad_token_id is None:

    tokenizer.pad_token_id = tokenizer.eos_token_id
    if model.config.pad_token_id is None:
        model.config.pad_token_id = tokenizer.pad_token_id

def format_prompt_enhanced(input_text, tokenizer):
    """
    Формирует промпт для модели, добавляя роль, инструкции, примеры (few-shot) и ожидаемый формат.
    """
    example_1_input = "Дата: 2023-01-10. Золото OHLCV: Open=1750.0, High=1755.0, Low=1748.0, Close=1752.0, Volume=800. Новости дня: Инфляция снижается / Курс доллара укрепляется / Рынок акций стабилен."
    example_1_output = "Метка: Sell\nОбоснование: Укрепление доллара и снижение инфляции обычно негативно сказываются на стоимости золота, поскольку оно становится менее привлекательным в качестве средства сохранения стоимости. Стабильный рынок акций также снижает спрос на безопасные активы."

    example_2_input = "Дата: 2023-01-11. Золото OHLCV: Open=1752.0, High=1760.0, Low=1751.0, Close=1758.0, Volume=1100. Новости дня: Геополитическая напряженность растет на Ближнем Востоке / Центральные банки рассматривают смягчение монетарной политики."
    example_2_output = "Метка: Buy\nОбоснование: Рост геополитической напряженности увеличивает спрос на золото как безопасный актив. Возможное смягчение монетарной политики центральных банков также способствует росту цен на золото из-за ожиданий увеличения ликвидности и инфляции."

    example_3_input = "Дата: 2023-01-12. Золото OHLCV: Open=1758.0, High=1759.0, Low=1757.0, Close=1758.0, Volume=900. Новости дня: Экономические данные смешанные / Нет значительных изменений на мировых рынках / Торги спокойные."
    example_3_output = "Метка: Hold\nОбоснование: Отсутствие явных экономических или геополитических триггеров и стабильные показатели OHLCV указывают на отсутствие причин для изменения текущей позиции. Рынок находится в состоянии неопределенности."

    system_prompt = (
        "Ты опытный финансовый аналитик, специализирующийся на рынке золота. Твоя задача — проанализировать "
        "предоставленные данные по цене золота (OHLCV) и новостям дня, а затем принять обоснованное инвестиционное решение: "
        "Buy (Покупка), Hold (Удержание) или Sell (Продажа).\n\n"
        "Твой анализ должен быть глубоким и содержать:\n"
        "1.  **Оценку ценовых движений и объема:** Как текущие цены (Open, High, Low, Close) и объем торгов (Volume) соотносятся "
        "с общим трендом или диапазоном? Есть ли признаки прорыва, консолидации или разворота? Не уделяй внимания объему.\n"
        "2.  **Анализ релевантности новостей:** Какие из новостей дня напрямую или косвенно влияют на цену золота? Например, новости "
        "о монетарной политике, инфляции, геополитической нестабильности, макроэкономических показателях США или крупнейших экономик могут быть "
        "важны для золота как защитного актива. Игнорируй новости, не имеющие отношения к рынку золота или общие корпоративные новости, "
        "если они не влияют на общие экономические настроения. Обращай внимания на молейшие сигналы для изменения цены на момент следующей торговой недели\n"
        "3.  **Обоснование решения:** Четко объясни, почему было выбрано именно это решение (Buy, Hold, Sell), ссылаясь на конкретные "
        "технические или фундаментальные факторы из предоставленных данных. Твое обоснование должно быть кратким, но информативным (2-4 предложения).\n\n"
        "Если данных недостаточно для принятия четкого решения, или если рынок находится в состоянии неопределенности, обоснуй, "
        "почему 'Hold' является наиболее разумной стратегией, и чего ты ожидаешь для изменения этого решения. Но старайся не злоупотреблять HOLD, если есть возможность, лучше использовать другие метки.\n\n"
        "Представь ответ в СТРОГОМ формате:\nМетка: [Hold/Buy/Sell]\nОбоснование: [Твоё подробное объяснение]"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Данные:\n{example_1_input}\n{example_1_output}"},
        {"role": "user", "content": f"Данные:\n{example_2_input}\n{example_2_output}"},
        {"role": "user", "content": f"Данные:\n{example_3_input}\n{example_3_output}"},
        {"role": "user", "content": f"Данные:\n{input_text}\n"}
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def extract_label_and_justification_improved(generated_text):
    original = generated_text
    label = "UNKNOWN"
    justification = "Не удалось извлечь обоснование."


    generated_text = re.sub(r"^(.*?)(Метка:|Рекомендация:|Buy|Sell|Hold)", r"\2", generated_text, flags=re.DOTALL | re.IGNORECASE)


    label_pattern = r"(?:Метка:|Рекомендация:|Decision:|Label:)\s*(buy|sell|hold)"
    label_match = re.search(label_pattern, generated_text, re.IGNORECASE)

    if label_match:
        label = label_match.group(1).upper()
        start_idx = label_match.end()
    else:
        for keyword in ["Buy", "Sell", "Hold"]:
            if keyword.lower() in generated_text.lower():
                label = keyword.upper()
                start_idx = generated_text.lower().find(keyword.lower()) + len(keyword)
                break
        else:
            for keyword in ["Buy", "Sell", "Hold"]:
                if keyword.lower() in generated_text[-100:].lower():
                    label = keyword.upper()
                    break

    if label != "UNKNOWN":
        after_label = generated_text[start_idx:]
        justification_end = re.search(r"(Buy|Sell|Hold|---|\[|$)", after_label, re.IGNORECASE)
        if justification_end:
            justification = after_label[:justification_end.start()].strip()
        else:
            justification = after_label.strip()
    else:
        justification = generated_text.strip()

    justification = re.sub(r"^(Обоснование:|Justification:)\s*", "", justification, flags=re.IGNORECASE).strip()
    return label, justification

generated_dataset = [
    {"input_text": "Дата: 2021-08-30. Золото OHLCV: Open=273.9, High=273.9, Low=273.9, Close=273.9, Volume=0. Новости дня: Recalls Fuel Disclosure Debate / Deutsche Telekom Defends / Japan's July Industrial Output Fell / U.S. Stocks End Mixed On Profit Concerns / PeopleFirst.com Acquires Giggo.com / Passion for Plasma Fuels TV Technology / Suzuki to Make Cars for GM / Group Criticizes Hong Kong Businessman / Jefferson Smurfit's Profit Rose Fivefold / FairMarket Names New CEO / Suharto to Be Charged With Fraud / New-Home Sales Rise Sharply / CMG Profit Rise Beats Expectations"},
]

generation_config = GenerationConfig(
    max_new_tokens=400,
    do_sample=False,
    num_beams=1,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
    num_return_sequences=1,
    repetition_penalty=1.1
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("\n--- Запуск предсказаний на наборе данных ---")
for i, item in enumerate(tqdm(generated_dataset, desc="Predicting")):
    input_text = item["input_text"]
    prompt = format_prompt_enhanced(input_text, tokenizer)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=model.config.max_position_embeddings).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            generation_config=generation_config,
        )

    generated_tokens_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    predicted_text = tokenizer.decode(generated_tokens_ids, skip_special_tokens=True).strip()

    cleaned_predicted_text = re.sub(r"^(Данные:\s*Дата:\s*\d{4}-\d{2}-\d{2}\.\s*Золото OHLCV:\s*Open=[\d\.]+, High=[\d\.]+, Low=[\d\.]+, Close=[\d\.]+, Volume=\d+\.\s*Новости дня:\s*(?:.|\n)*?)?(Метка:|Обоснование:)", r"\2", predicted_text, flags=re.IGNORECASE|re.DOTALL).strip()
    cleaned_predicted_text = re.sub(r"^(Новости дня:\s*(?:.|\n)*?)?(Метка:|Обоснование:)", r"\2", cleaned_predicted_text, flags=re.IGNORECASE|re.DOTALL).strip()

    cleaned_predicted_text = re.sub(r"^(Дата:\s*\d{4}-\d{2}-\d{2}\.\s*Золото OHLCV:.*?Новости дня:.*?Метка:\s*(?:Sell|Buy|Hold)\s*Обоснование:.*?)", "", cleaned_predicted_text, flags=re.IGNORECASE|re.DOTALL).strip()

    predicted_label, justification = extract_label_and_justification_improved(cleaned_predicted_text)

    print(f"\n--- Пример {i+1} ---")
    print(f"[Input]: {input_text}")
    print(f"[Raw Predicted Text]: {cleaned_predicted_text}")
    print(f"[Predicted Label]: {predicted_label}")
    print(f"[Justification]: {justification}")
    print("-" * 60)


--- Запуск предсказаний на наборе данных ---


Predicting:  10%|█         | 1/10 [00:46<06:58, 46.46s/it]


--- Пример 1 ---
[Input]: Дата: 2000-08-30. Золото OHLCV: Open=273.9, High=273.9, Low=273.9, Close=273.9, Volume=0. Новости дня: Recalls Fuel Disclosure Debate / Deutsche Telekom Defends / Japan's July Industrial Output Fell / U.S. Stocks End Mixed On Profit Concerns / PeopleFirst.com Acquires Giggo.com / Passion for Plasma Fuels TV Technology / Suzuki to Make Cars for GM / Group Criticizes Hong Kong Businessman / Jefferson Smurfit's Profit Rose Fivefold / FairMarket Names New CEO / Suharto to Be Charged With Fraud / New-Home Sales Rise Sharply / CMG Profit Rise Beats Expectations
[Raw Predicted Text]: Метка: Hold
Обоснование: Абсолютный отсутствие торгового объема и цены золота на уровне 273.9 USD/ounce делают любую реакцию на новости дня незначительной. В таких условиях наиболее разумная стратегия - "hold". Рынок находится в состоянии неопределенности.
[Predicted Label]: HOLD
[Justification]: Абсолютный отсутствие торгового объема и цены золота на уровне 273.9 USD/ounce делают любую

Predicting:  20%|██        | 2/10 [01:12<04:33, 34.25s/it]


--- Пример 2 ---
[Input]: Дата: 2000-08-31. Золото OHLCV: Open=274.8, High=278.3, Low=274.8, Close=278.3, Volume=0. Новости дня: Credit Suisse's Wheat Faces Deal / Regulators Turn to Early Reports / NTT to Close Offer for Verio / U.S. Markets End Higher On Tech, Financial Stocks / Suharto's Corruption Trial Is Adjourned / Datops, NetResult Team Up for Games / Online Retail Sales Rise 5.3% / After Paper Era, What Comes Next? / Hitachi to Import Parts From China / France Unveils Tax-Cut Program / Sanyo Raises Sales, Profit Forecasts / Where Are They Now? / Ford Trims Light-Truck Output / Lucent Sets Avaya Spinoff Terms
[Raw Predicted Text]: Метка: Hold
Обоснование: В данный момент нет явных экономических или геополитических триггеров, которые могли бы повлиять на цену золота. Предоставившиеся новости не имеют прямого отношения к цене золота, а объём торгов низкий. Поэтому, в отсутствие явных причин для изменения текущей позиции, "Hold" является наиболее разумной стратегией. Рынок находи

...


--- Пример 8 ---
[Input]: Дата: 2000-09-13. Золото OHLCV: Open=272.8, High=272.8, Low=272.8, Close=272.8, Volume=0. Новости дня: Netscape to Launch New Portal / U.S. Markets End Mixed As Profit Jitters Continue / Microsoft to Acquire Music-Search Site / North Korea Nears Admission to Bank / Judge Dismisses Suit UBS Filed Over Fund / Commingling Two Long Histories / World Bank Rethinks Poverty Strategy
[Raw Predicted Text]: Метка: Hold
Обоснование: Хотя центральные банки продолжают рассматривать смягчение монетарной политики, необходимые условия для этого ещё не созданы. Высокие процентные ставки в США продолжают создавать неопределенность, что может повлиять на цену золота. Прогнозы экономического роста в Европе остаются оптимистическими, что может привести к уменьшению спроса на золото как безопасный актив. Рынок находится в состоянии неопределенности.
[Predicted Label]: HOLD
[Justification]: Хотя центральные банки продолжают рассматривать смягчение монетарной политики, необходимые у

Predicting:  90%|█████████ | 9/10 [06:00<00:37, 37.62s/it]


--- Пример 9 ---
[Input]: Дата: 2000-09-20. Золото OHLCV: Open=269.0, High=269.0, Low=269.0, Close=269.0, Volume=0. Новости дня: Critics Seek Changes In Broker Reporting / Dell Plans Internet Marketplace / Restructuring Plan Sinks Informix / Two Firms Give SEC An Auditing Proposal / Europeans Debate Future of Work / E.Biscom May Make More Deals / Olympic Enforcers Scour Net for Clips
[Raw Predicted Text]: Метка: Hold
Обоснование: В данной ситуации нет явных экономических или геополитических триггеров, которые могли повлиять на цену золота. Предоставившиеся новости не имеют прямого влияния на цену золота, поскольку они не связаны с экономическими или геополитическими факторами, которые могут повлиять на цену золота. Рынок находится в состоянии неопределенности, и лучшей стратегией будет продолжать держать текущую позицию.
[Predicted Label]: HOLD
[Justification]: В данной ситуации нет явных экономических или геополитических триггеров, которые могли повлиять на цену золота. Предоставивши

Predicting: 100%|██████████| 10/10 [06:25<00:00, 38.59s/it]


--- Пример 10 ---
[Input]: Дата: 2000-09-21. Золото OHLCV: Open=270.3, High=270.3, Low=270.3, Close=270.3, Volume=0. Новости дня: Liberate Posts Quarterly Results / Invensys Expects Baan to Recover / State-Run Bank May Oversee Daewoo Sale / Full Text of Greenspan Testimony / Marconi to Buy Mariposa for $268 Million / Engage Cuts Work Force by 13%
[Raw Predicted Text]: Метка: Hold
Обоснование: В данный момент нет достаточных факторов для принятия решения о покупке или продаже золота. Лишь несколько новостей дня имеют отношение к золоту, и они не имеют существенного влияния на цену металла. Помимо этого, низкий объём торгов и высокая цена золота указывают на отсутствие значимых изменений в ближайшее время. Рынок находится в состоянии неопределенности.
[Predicted Label]: HOLD
[Justification]: В данный момент нет достаточных факторов для принятия решения о покупке или продаже золота. Лишь несколько новостей дня имеют отношение к золоту, и они не имеют существенного влияния на цену металла